In [2]:
import os
import sys
from torch.utils.data import DataLoader

# Add project root to PYTHONPATH so `utils` can be imported
workspace_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)

from utils.dataset import RoWSFormerDataset

# Set dataset directory (DIV2K validation set)
train_hr_dir = os.path.join(workspace_root, "data", "DIV2K_train_HR", "DIV2K_train_HR")
valid_hr_dir = os.path.join(workspace_root, "data", "DIV2K_valid_HR", "DIV2K_valid_HR")

# Create dataset and dataloader for testing
train_dataset = RoWSFormerDataset(img_dir=train_hr_dir, img_size=128, bit_length=64)
val_dataset = RoWSFormerDataset(img_dir=valid_hr_dir, img_size=128, bit_length=64)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)


In [3]:
# Load and inspect a sample
sample = train_dataset[0]
print(f"Sample type: {type(sample)}")
print(f"Number of items: {len(sample)}")

# Unpack the tuple (likely image, watermark)
image, watermark = sample
print(f"Image shape: {image.shape}")
print(f"Watermark shape: {watermark.shape}")

Sample type: <class 'tuple'>
Number of items: 2
Image shape: torch.Size([3, 128, 128])
Watermark shape: torch.Size([64])


In [4]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import math


In [5]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [6]:
class PatchEmbedding(nn.Module):
    """
    Converts image to patch embeddings.
    
    Args:
        in_channels: Input channels (3 for RGB)
        embed_dim: Embedding dimension (C)
        patch_size: Size of each patch (P)
    """
    def __init__(self, in_channels=3, embed_dim=64, patch_size=4):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        
        # Simple conv layer to extract features
        # From paper: "we first apply a 3×3 convolutional layer"
        self.conv = nn.Conv2d(in_channels, embed_dim, 
                             kernel_size=3, padding=1, stride=1)
    
    # convolution ko feed forward layer jastai use bhacha
    def forward(self, x):
        """
        Args:
            x: (B, 3, H, W)
        Returns:
            features: (B, C, H, W)
        """
        # Extract features
        features = self.conv(x)  # (B, C, H, W)
        return features

# Test
patch_embed = PatchEmbedding(in_channels=3, embed_dim=64, patch_size=4)
test_img = torch.randn(1, 3, 128, 128)
features = patch_embed(test_img)
print(f"Input shape: {test_img.shape}")
print(f"Output shape: {features.shape}")
print("✓ Patch embedding working!")

Input shape: torch.Size([1, 3, 128, 128])
Output shape: torch.Size([1, 64, 128, 128])
✓ Patch embedding working!


In [7]:
class WatermarkEncoder(nn.Module):
    """
    Encodes watermark bits and expands them spatially.

    Process:
    1. Bits (L,) → Linear → (L1,)
    2. Reshape → (L2, L2)
    3. Upsample → (H, W) to match image features
    4. Conv → (C1,) channels

    Args:
        watermark_length: Number of bits (64)
        embed_dim: Output channels (C1)
        image_size: Target spatial size (128)
    """
    def __init__(self, watermark_length=64, embed_dim=32, image_size=128):
        super().__init__()
        self.watermark_length = watermark_length
        self.embed_dim = embed_dim
        self.image_size = image_size

        # From paper: "Men passes through a linear layer that produces
        # an output vector of length L1. This vector is then reshaped
        # into a matrix of size L2×L2."

        # We'll use a simple approach: bits → small spatial map → upsample
        # L1 = 16*16 = 256 (for example)
        L1 = 256
        L2 = 16  # sqrt(256)

        self.linear = nn.Linear(watermark_length, L1)
        self.L2 = L2

        # Conv to increase channels after upsampling
        self.conv = nn.Conv2d(1, embed_dim, kernel_size=3, padding=1)

    def forward(self, watermark, target_size):
        """
        Args:
            watermark: (B, L) - binary watermark bits
            target_size: (H, W) - size to match
        Returns:
            (B, C1, H, W) - spatially expanded watermark features
        """
        B = watermark.shape[0]

        # Linear projection
        x = self.linear(watermark)  # (B, L1)

        # Reshape to spatial
        x = x.view(B, 1, self.L2, self.L2)  # (B, 1, L2, L2)

        # Upsample to target size using nearest neighbor
        # From paper: "nearest-neighbor interpolation method is used"
        x = F.interpolate(x, size=target_size, mode='nearest')  # (B, 1, H, W)

        # Increase channels
        x = self.conv(x)  # (B, C1, H, W)

        return x

# Test
wm_encoder = WatermarkEncoder(watermark_length=64, embed_dim=32)
test_wm = torch.randint(0, 2, (1, 64), dtype=torch.float32)
wm_features = wm_encoder(test_wm, target_size=(128, 128))
print(f"Watermark input shape: {test_wm.shape}")
print(f"Watermark features shape: {wm_features.shape}")
print("✓ Watermark encoder working!")

Watermark input shape: torch.Size([1, 64])
Watermark features shape: torch.Size([1, 32, 128, 128])
✓ Watermark encoder working!


In [8]:
def window_partition(x, window_size):
    """
    Partition feature map into non-overlapping windows.

    Args:
        x: (B, C, H, W)
        window_size: Window size M
    Returns:
        windows: (B*num_windows, C, M, M)
    """
    B, C, H, W = x.shape
    x = x.view(B, C, H // window_size, window_size, W // window_size, window_size)
    windows = x.permute(0, 2, 4, 1, 3, 5).contiguous() # (B, num_windows_H, num_windows_W, C, M, M)

    # -1 is a cheesy syntax for saying "infer this dimension" 
    windows = windows.view(-1, C, window_size, window_size) # (B*num_windows, C, M, M)
    return windows

def window_reverse(windows, window_size, H, W):
    """
    Reverse window partition.

    Args:
        windows: (B*num_windows, C, M, M)
        window_size: Window size M
        H, W: Original height and width
    Returns:
        x: (B, C, H, W)
    """
    B_w, C, M, M = windows.shape
    B = B_w // ((H // window_size) * (W // window_size))
    x = windows.view(B, H // window_size, W // window_size, C, window_size, window_size)
    x = x.permute(0, 3, 1, 4, 2, 5).contiguous()
    x = x.view(B, C, H, W)
    return x

# Test window operations
test_x = torch.randn(1, 64, 128, 128)
windows = window_partition(test_x, window_size=8)
print(f"Original shape: {test_x.shape}")
print(f"Windows shape: {windows.shape}")
print(f"Number of windows: {(128//8) * (128//8)} = 256")

# Reverse
reconstructed = window_reverse(windows, window_size=8, H=128, W=128)
print(f"Reconstructed shape: {reconstructed.shape}")
print(f"Reconstruction error: {(test_x - reconstructed).abs().max():.6f}")
print("✓ Window partition working!")

Original shape: torch.Size([1, 64, 128, 128])
Windows shape: torch.Size([256, 64, 8, 8])
Number of windows: 256 = 256
Reconstructed shape: torch.Size([1, 64, 128, 128])
Reconstruction error: 0.000000
✓ Window partition working!


In [9]:
class WindowAttention(nn.Module):
    """
    Window-based multi-head self-attention (W-MSA).

    Implements the core attention mechanism used in Swin Transformer.
    Computes attention within local windows for efficiency.

    Args:
        dim: Input dimension (number of channels)
        window_size: Window size (M)
        num_heads: Number of attention heads
    """
    def __init__(self, dim, window_size=8, num_heads=4):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        # QKV projection
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        """
        Args:
            x: (B*num_windows, M*M, C) - flattened windows
        Returns:
            (B*num_windows, M*M, C)
        """
        B_, N, C = x.shape

        # Generate Q, K, V
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B_, num_heads, N, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Attention: Q @ K^T / sqrt(d)
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B_, num_heads, N, N)
        attn = F.softmax(attn, dim=-1)

        # Attention @ V
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)

        return x

# Test
window_attn = WindowAttention(dim=64, window_size=8, num_heads=4)
test_input = torch.randn(256, 64, 64)  # (B*num_windows, M*M, C)
output = window_attn(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {output.shape}")
print("✓ Window attention working!")

Input shape: torch.Size([256, 64, 64])
Output shape: torch.Size([256, 64, 64])
✓ Window attention working!


In [10]:
class SwinTransformerBlock(nn.Module):
    """
    Swin Transformer Block with window-based attention.

    From paper Eq. (3):
    X̂ˡ = W-MSA(LN(Xˡ⁻¹)) + Xˡ⁻¹
    Xˡ = MLP(LN(X̂ˡ)) + X̂ˡ

    Args:
        dim: Channel dimension
        num_heads: Number of attention heads
        window_size: Window size for attention
        shift_size: Shift size for shifted window attention (0 for W-MSA)
    """
    def __init__(self, dim, num_heads=4, window_size=8, shift_size=0):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size

        # Normalization
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

        # Window attention
        self.attn = WindowAttention(dim, window_size, num_heads)

        # MLP: 2-layer with GELU
        mlp_hidden_dim = int(dim * 4)  # Standard expansion ratio
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, dim)
        )

    def forward(self, x, H, W):
        """
        Args:
            x: (B, C, H, W)
            H, W: Spatial dimensions
        Returns:
            (B, C, H, W)
        """
        B, C, H, W = x.shape
        shortcut = x

        # Reshape to (B, H*W, C) for attention
        x = x.flatten(2).transpose(1, 2)  # (B, H*W, C)

        # Cyclic shift (for shifted   window attention)
        if self.shift_size > 0:
            x = x.view(B, H, W, C)
            x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
            x = x.view(B, H * W, C)

        # Partition into windows
        x = x.view(B, H, W, C)
        x_windows = window_partition(
            x.permute(0, 3, 1, 2).contiguous(),
            self.window_size
        )  # (B*num_windows, C, M, M)

        # Flatten windows for attention
        x_windows = x_windows.flatten(2).transpose(1, 2)  # (B*num_windows, M*M, C)

        # W-MSA
        x_windows = self.norm1(x_windows)
        attn_windows = self.attn(x_windows)  # (B*num_windows, M*M, C)

        # Reverse window partition
        attn_windows = attn_windows.transpose(1, 2).view(-1, C, self.window_size, self.window_size)
        x = window_reverse(attn_windows, self.window_size, H, W)  # (B, C, H, W)

        # Reverse cyclic shift
        if self.shift_size > 0:
            x = x.permute(0, 2, 3, 1).contiguous()  # (B, H, W, C)
            x = torch.roll(x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
            x = x.permute(0, 3, 1, 2).contiguous()  # (B, C, H, W)

        # First residual connection
        x = shortcut + x

        # MLP block
        shortcut = x
        x = x.flatten(2).transpose(1, 2)  # (B, H*W, C)
        x = self.norm2(x)
        x = self.mlp(x)
        x = x.transpose(1, 2).view(B, C, H, W)

        # Second residual connection
        x = shortcut + x

        return x

# Test
swin_block = SwinTransformerBlock(dim=64, num_heads=4, window_size=8)
test_x = torch.randn(1, 64, 128, 128)
output = swin_block(test_x, H=128, W=128)
print(f"Input shape: {test_x.shape}")
print(f"Output shape: {output.shape}")
print("✓ Swin Transformer Block working!")

Input shape: torch.Size([1, 64, 128, 128])
Output shape: torch.Size([1, 64, 128, 128])
✓ Swin Transformer Block working!


In [11]:
class LocallyChannelEnhancedBlock(nn.Module):
    """
    Locally-Channel Enhanced Block (LCEB).

    Captures local features with depthwise conv and channel attention.

    From paper:
    1. Linear projection (expand)
    2. Depthwise 3×3 conv (local features)
    3. Linear projection (compress)
    4. Channel attention (reweight channels)

    Args:
        dim: Channel dimension
        expansion: Channel expansion ratio (default 2)
    """
    def __init__(self, dim, expansion=2):
        super().__init__()
        hidden_dim = int(dim * expansion)

        # Linear projection to expand channels
        self.expand = nn.Conv2d(dim, hidden_dim, kernel_size=1)

        # Depthwise convolution for local features
        # From paper: "3×3 depthwise convolution to capture local features"
        self.dwconv = nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3,
                               padding=1, groups=hidden_dim)

        # Linear projection to compress back
        self.compress = nn.Conv2d(hidden_dim, dim, kernel_size=1)

        # Channel attention
        # From paper: "a pooling layer followed by a fully connected layer
        # is used to compute attention weights for each channel"
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.channel_attn = nn.Sequential(
            nn.Linear(dim, dim // 4),
            nn.ReLU(inplace=True),
            nn.Linear(dim // 4, dim),
            nn.Sigmoid()
        )

        self.act = nn.GELU()

    def forward(self, x):
        """
        Args:
            x: (B, C, H, W)
        Returns:
            bias: (B, C, H, W) - local and channel-enhanced features
        """
        identity = x

        # Expand
        x = self.expand(x)
        x = self.act(x)

        # Depthwise conv for local features
        x = self.dwconv(x)
        x = self.act(x)

        # Compress
        x = self.compress(x)

        # Channel attention
        B, C, H, W = x.shape
        channel_weights = self.pool(x).view(B, C)  # (B, C)
        channel_weights = self.channel_attn(channel_weights).view(B, C, 1, 1)  # (B, C, 1, 1)

        # Apply channel attention
        x = x * channel_weights

        # Add to identity (residual)
        return identity + x

# Test
lceb = LocallyChannelEnhancedBlock(dim=64)
test_x = torch.randn(1, 64, 128, 128)
output = lceb(test_x)
print(f"Input shape: {test_x.shape}")
print(f"Output shape: {output.shape}")
print("✓ Locally-Channel Enhanced Block working!")

Input shape: torch.Size([1, 64, 128, 128])
Output shape: torch.Size([1, 64, 128, 128])
✓ Locally-Channel Enhanced Block working!


In [12]:
class LCESTB(nn.Module):
    """
    Locally-Channel Enhanced Swin Transformer Block.

    Combines:
    1. Swin Transformer Block (global context via windowed attention)
    2. Locally-Channel Enhanced Block (local features + channel attention)

    This is the core building block of RoWSFormer encoder and decoder.

    Args:
        dim: Channel dimension
        num_heads: Number of attention heads
        window_size: Window size for attention
        shift_size: Whether to use shifted windows (0 or window_size//2)
    """
    def __init__(self, dim, num_heads=4, window_size=8, shift_size=0):
        super().__init__()

        # From paper: "LESTB consists of two main parts"
        # Part 1: Swin Transformer Block
        self.swin_block = SwinTransformerBlock(
            dim=dim,
            num_heads=num_heads,
            window_size=window_size,
            shift_size=shift_size
        )

        # Part 2: Locally-Channel Enhanced Block
        self.lce_block = LocallyChannelEnhancedBlock(dim=dim)

    def forward(self, x, H, W):
        """
        Args:
            x: (B, C, H, W)
            H, W: Spatial dimensions
        Returns:
            (B, C, H, W)
        """
        # First: Swin block for global modeling
        x = self.swin_block(x, H, W)

        # Second: LCE block for local and channel modeling
        x = self.lce_block(x)

        return x

# Test
lcestb = LCESTB(dim=64, num_heads=4, window_size=8)
test_x = torch.randn(1, 64, 128, 128)
output = lcestb(test_x, H=128, W=128)
print(f"Input shape: {test_x.shape}")
print(f"Output shape: {output.shape}")
print("✓ LCESTB working!")
print("\n🎯 This is the core component of RoWSFormer!")

Input shape: torch.Size([1, 64, 128, 128])
Output shape: torch.Size([1, 64, 128, 128])
✓ LCESTB working!

🎯 This is the core component of RoWSFormer!


In [13]:
class FrequencyEnhancedBlock(nn.Module):
    """
    Frequency-Enhanced Block using simplified frequency modeling.

    From paper:
    - Applies DCT to extract frequency features
    - Uses FC layer to compute frequency attention weights

    Simplified implementation:
    - Uses learnable frequency filters instead of explicit DCT
    - Applies channel-wise frequency attention

    Args:
        dim: Channel dimension
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

        # Global pooling to get frequency statistics
        self.pool = nn.AdaptiveAvgPool2d(1)

        # FC layer to compute frequency attention weights
        # From paper: "a simple fully connected (FC) layer is used to
        # compute the frequency domain attention weights"
        self.fc = nn.Sequential(
            nn.Linear(dim, dim // 4),
            nn.ReLU(inplace=True),
            nn.Linear(dim // 4, dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        """
        Args:
            x: (B, C, H, W)
        Returns:
            (B, C, H, W) - frequency-enhanced features
        """
        B, C, H, W = x.shape

        # Global pooling (approximates frequency spectrum statistics)
        freq_stats = self.pool(x).view(B, C)  # (B, C)

        # Compute frequency attention weights
        freq_weights = self.fc(freq_stats).view(B, C, 1, 1)  # (B, C, 1, 1)

        # Apply frequency attention
        out = x * freq_weights

        return out

# Test
feb = FrequencyEnhancedBlock(dim=64)
test_x = torch.randn(1, 64, 16, 16)  # Smaller spatial size (bottleneck)
output = feb(test_x)
print(f"Input shape: {test_x.shape}")
print(f"Output shape: {output.shape}")
print("✓ Frequency-Enhanced Block working!")

Input shape: torch.Size([1, 64, 16, 16])
Output shape: torch.Size([1, 64, 16, 16])
✓ Frequency-Enhanced Block working!


In [14]:
class GlobalTransformerBlock(nn.Module):
    """
    Standard Transformer block with global attention.

    Used at the bottleneck where spatial dimensions are small,
    so global attention is computationally feasible.

    From paper Eq. (4):
    X̂ˡ = MSA(LN(Xˡ⁻¹)) + Xˡ⁻¹
    Xˡ = MLP(LN(X̂ˡ)) + X̂ˡ

    Args:
        dim: Channel dimension
        num_heads: Number of attention heads
    """
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        # Normalization
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

        # QKV projection
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

        # MLP
        mlp_hidden_dim = int(dim * 4)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_hidden_dim, dim)
        )

    def forward(self, x):
        """
        Args:
            x: (B, C, H, W)
        Returns:
            (B, C, H, W)
        """
        B, C, H, W = x.shape
        shortcut = x

        # Reshape to (B, N, C) where N = H*W
        x = x.flatten(2).transpose(1, 2)  # (B, H*W, C)
        N = x.shape[1]

        # Multi-head self-attention
        x = self.norm1(x)
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, num_heads, N, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = F.softmax(attn, dim=-1)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)

        # Reshape back and residual
        x = x.transpose(1, 2).view(B, C, H, W)
        x = shortcut + x

        # MLP block
        shortcut = x
        x = x.flatten(2).transpose(1, 2)  # (B, N, C)
        x = self.norm2(x)
        x = self.mlp(x)
        x = x.transpose(1, 2).view(B, C, H, W)

        # Residual
        x = shortcut + x

        return x

# Test
global_block = GlobalTransformerBlock(dim=64, num_heads=4)
test_x = torch.randn(1, 64, 16, 16)
output = global_block(test_x)
print(f"Input shape: {test_x.shape}")
print(f"Output shape: {output.shape}")
print("✓ Global Transformer Block working!")

Input shape: torch.Size([1, 64, 16, 16])
Output shape: torch.Size([1, 64, 16, 16])
✓ Global Transformer Block working!


In [15]:
class FETB(nn.Module):
    """
    Frequency-Enhanced Transformer Block.

    Combines:
    1. Global Transformer Block (capture global context)
    2. Frequency-Enhanced Block (frequency domain modeling)

    Used at the bottleneck of the encoder/decoder.

    Args:
        dim: Channel dimension
        num_heads: Number of attention heads
        num_blocks: Number of Transformer blocks (default 2)
    """
    def __init__(self, dim, num_heads=4, num_blocks=2):
        super().__init__()

        # Multiple Transformer blocks
        self.blocks = nn.ModuleList([
            GlobalTransformerBlock(dim, num_heads)
            for _ in range(num_blocks)
        ])

        # Frequency enhancement
        self.freq_enhance = FrequencyEnhancedBlock(dim)

    def forward(self, x):
        """
        Args:
            x: (B, C, H, W)
        Returns:
            (B, C, H, W)
        """
        # Apply Transformer blocks
        for block in self.blocks:
            x = block(x)

        # Apply frequency enhancement
        x = self.freq_enhance(x)

        return x

# Test
fetb = FETB(dim=128, num_heads=4, num_blocks=2)
test_x = torch.randn(1, 128, 16, 16)
output = fetb(test_x)
print(f"Input shape: {test_x.shape}")
print(f"Output shape: {output.shape}")
print("✓ FETB working!")
print("\n🎯 This enhances robustness through frequency modeling!")

Input shape: torch.Size([1, 128, 16, 16])
Output shape: torch.Size([1, 128, 16, 16])
✓ FETB working!

🎯 This enhances robustness through frequency modeling!


In [16]:
class Encoder(nn.Module):
    """
    RoWSFormer Encoder.

    Embeds watermark into cover image using U-Net architecture with:
    - LCESTB blocks for feature extraction
    - FETB at bottleneck
    - Skip connections
    - Watermark feature fusion

    Args:
        in_channels: Input channels (3 for RGB)
        base_dim: Base channel dimension (C)
        num_stages: Number of down/up sampling stages (K)
        watermark_length: Watermark bit length (64)
    """
    def __init__(self, in_channels=3, base_dim=64, num_stages=3, watermark_length=64):
        super().__init__()
        self.num_stages = num_stages
        self.base_dim = base_dim

        # Initial feature extraction
        self.patch_embed = PatchEmbedding(in_channels, base_dim)

        # Watermark encoder
        self.watermark_encoder = WatermarkEncoder(
            watermark_length=watermark_length,
            embed_dim=base_dim // 2  # C1 in paper
        )

        # Downsampling stages
        self.down_blocks = nn.ModuleList()
        self.down_samples = nn.ModuleList()

        for i in range(num_stages):
            dim = base_dim * (2 ** i)

            # LCESTB (alternating between W-MSA and SW-MSA)
            self.down_blocks.append(
                nn.Sequential(
                    LCESTB(dim, num_heads=4, window_size=8, shift_size=0),
                    LCESTB(dim, num_heads=4, window_size=8, shift_size=4)
                )
            )

            # Downsampling (except last stage)
            if i < num_stages - 1:
                self.down_samples.append(
                    nn.Conv2d(dim, dim * 2, kernel_size=4, stride=2, padding=1)
                )

        # Bottleneck with FETB
        bottleneck_dim = base_dim * (2 ** (num_stages - 1))
        self.bottleneck = FETB(bottleneck_dim, num_heads=4, num_blocks=2)

        # Upsampling stages
        self.up_samples = nn.ModuleList()
        self.up_blocks = nn.ModuleList()

        for i in range(num_stages - 1, -1, -1):
            dim = base_dim * (2 ** i)

            # Upsampling (except first upsampling stage)
            if i < num_stages - 1:
                self.up_samples.append(
                    nn.ConvTranspose2d(dim * 2, dim, kernel_size=2, stride=2)
                )

            # Concatenation: upsampled + skip + watermark
            # From paper: concatenated with feature map from downsampling
            # and watermark feature map
            concat_dim = dim * 2 + base_dim // 2 if i < num_stages - 1 else dim

            self.up_blocks.append(
                nn.Sequential(
                    nn.Conv2d(concat_dim, dim, kernel_size=3, padding=1),
                    LCESTB(dim, num_heads=4, window_size=8, shift_size=0),
                    LCESTB(dim, num_heads=4, window_size=8, shift_size=4)
                )
            )

        # Output layer
        self.output = nn.Conv2d(base_dim, in_channels, kernel_size=3, padding=1)

    def forward(self, image, watermark):
        """
        Args:
            image: (B, 3, H, W) - cover image
            watermark: (B, L) - watermark bits
        Returns:
            watermarked: (B, 3, H, W) - watermarked image
        """
        B, C, H, W = image.shape

        # Initial features
        x = self.patch_embed(image)  # (B, C, H, W)

        # Downsampling
        skip_connections = []
        current_h, current_w = H, W

        for i, (block, downsample) in enumerate(
            zip(self.down_blocks, self.down_samples + [None])
        ):
            x = block[0](x, current_h, current_w)
            x = block[1](x, current_h, current_w)
            skip_connections.append(x)

            if downsample is not None:
                x = downsample(x)
                current_h, current_w = current_h // 2, current_w // 2

        # Bottleneck
        x = self.bottleneck(x)

        # Upsampling
        skip_connections = skip_connections[::-1]  # Reverse

        for i, (upsample, block) in enumerate(
            zip([None] + list(self.up_samples), self.up_blocks)
        ):
            if upsample is not None:
                x = upsample(x)
                current_h, current_w = current_h * 2, current_w * 2

            # Get watermark features for this resolution
            wm_features = self.watermark_encoder(
                watermark,
                target_size=(current_h, current_w)
            )

            # Concatenate: upsampled + skip + watermark
            if i > 0:
                x = torch.cat([x, skip_connections[i], wm_features], dim=1)
            else:
                x = skip_connections[i]  # First iteration

            x = block[0](x)  # Conv
            x = block[1](x, current_h, current_w)  # LCESTB 1
            x = block[2](x, current_h, current_w)  # LCESTB 2

        # Output (perturbation ΔI)
        delta = self.output(x)  # (B, 3, H, W)

        # Residual: I_em = I_co + ΔI
        watermarked = image + delta

        # Clamp to [0, 1]
        watermarked = torch.clamp(watermarked, 0, 1)

        return watermarked

# Test
print("Building encoder... (this may take a moment)")
encoder = Encoder(
    in_channels=3,
    base_dim=32,  # Reduced for memory
    num_stages=2,  # Reduced for memory
    watermark_length=64
).to(device)

# Count parameters
num_params = sum(p.numel() for p in encoder.parameters())
print(f"Encoder parameters: {num_params:,}")

# Test forward pass
with torch.no_grad():
    test_img = torch.randn(1, 3, 128, 128).to(device)
    test_wm = torch.randint(0, 2, (1, 64), dtype=torch.float32).to(device)
    watermarked = encoder(test_img, test_wm)
    print(f"Input shape: {test_img.shape}")
    print(f"Watermarked shape: {watermarked.shape}")
    print(f"Output range: [{watermarked.min():.3f}, {watermarked.max():.3f}]")

print("✓ Encoder working!")

Building encoder... (this may take a moment)
Encoder parameters: 573,939
Input shape: torch.Size([1, 3, 128, 128])
Watermarked shape: torch.Size([1, 3, 128, 128])
Output range: [0.000, 1.000]
✓ Encoder working!


In [25]:
import torch.nn.functional as F

class NoiseLayer(nn.Module):
    """
    Differentiable noise layer for training robustness.
    
    Implements various geometric and non-geometric attacks.
    All operations are differentiable to enable end-to-end training.
    
    Args:
        attack_types: List of attack types to use
    """
    def __init__(self, attack_types=None):
        super().__init__()
        
        if attack_types is None:
            # Default: use all attacks
            self.attack_types = [
                'none',  # Sometimes no attack
                'gaussian_noise',
                'gaussian_blur',
                'cropout',
                'dropout',
            ]
        else:
            self.attack_types = attack_types
    
    def gaussian_noise(self, x, sigma=None):
        """
        Add Gaussian noise.
        From paper: σ ∈ [0.001, 0.04] during training
        """
        if sigma is None:
            sigma = torch.rand(1).item() * 0.039 + 0.001
        
        noise = torch.randn_like(x) * sigma
        return torch.clamp(x + noise, 0, 1)
    
    def gaussian_blur(self, x, kernel_size=5):
        """
        Apply Gaussian blur.
        Approximated using average pooling for differentiability.
        """
        # Simple average blur (differentiable)
        kernel = torch.ones(1, 1, kernel_size, kernel_size) / (kernel_size ** 2)
        kernel = kernel.to(x.device)
        
        # Apply to each channel
        B, C, H, W = x.shape
        x_blur = []
        for c in range(C):
            blurred = F.conv2d(
                x[:, c:c+1], 
                kernel, 
                padding=kernel_size//2
            )
            x_blur.append(blurred)
        
        return torch.cat(x_blur, dim=1)
    
    def cropout(self, x, original, ratio=None):
        """
        Cropout attack: replace random region with original.
        From paper: ratio ∈ [0.1, 0.5] during training
        
        Args:
            x: Watermarked image
            original: Original cover image
            ratio: Fraction of image to crop
        """
        if ratio is None:
            ratio = torch.rand(1).item() * 0.4 + 0.1  # [0.1, 0.5]
        
        B, C, H, W = x.shape
        
        # Random crop size
        crop_h = int(H * ratio ** 0.5)
        crop_w = int(W * ratio ** 0.5)
        
        # Random position
        top = torch.randint(0, H - crop_h + 1, (1,)).item()
        left = torch.randint(0, W - crop_w + 1, (1,)).item()
        
        # Create mask
        mask = torch.ones_like(x)
        mask[:, :, top:top+crop_h, left:left+crop_w] = 0
        
        # Apply cropout
        return x * mask + original * (1 - mask)
    
    def dropout(self, x, original, ratio=None):
        """
        Dropout attack: replace random pixels with original.
        From paper: ratio ∈ [0.2, 0.6] during training
        
        Args:
            x: Watermarked image
            original: Original cover image
            ratio: Fraction of pixels to drop
        """
        if ratio is None:
            ratio = torch.rand(1).item() * 0.4 + 0.2  # [0.2, 0.6]
        
        # Random mask
        mask = (torch.rand_like(x) > ratio).float()
        
        # Apply dropout
        return x * mask + original * (1 - mask)
    
    def forward(self, x, original=None, attack_type=None):
        """
        Apply random attack.
        
        Args:
            x: Watermarked image (B, 3, H, W)
            original: Original image (needed for cropout/dropout)
            attack_type: Specific attack to use (random if None)
        Returns:
            attacked: (B, 3, H, W)
        """
        if attack_type is None:
            attack_type = np.random.choice(self.attack_types)
        
        if attack_type == 'none':
            return x
        elif attack_type == 'gaussian_noise':
            return self.gaussian_noise(x)
        elif attack_type == 'gaussian_blur':
            return self.gaussian_blur(x)
        elif attack_type == 'cropout':
            if original is None:
                return x
            return self.cropout(x, original)
        elif attack_type == 'dropout':
            if original is None:
                return x
            return self.dropout(x, original)
        else:
            return x

# Test
noise_layer = NoiseLayer()
test_img = torch.rand(1, 3, 128, 128)
test_original = torch.rand(1, 3, 128, 128)

print("Testing attacks:")
for attack in ['gaussian_noise', 'gaussian_blur', 'cropout', 'dropout']:
    attacked = noise_layer(test_img, test_original, attack_type=attack)
    diff = (test_img - attacked).abs().mean()
    print(f"  {attack:20s}: Mean diff = {diff:.4f}")

print("✓ Noise layer working!")

Testing attacks:
  gaussian_noise      : Mean diff = 0.0203
  gaussian_blur       : Mean diff = 0.2445
  cropout             : Mean diff = 0.1445
  dropout             : Mean diff = 0.1422
✓ Noise layer working!


In [18]:
class Decoder(nn.Module):
    """
    RoWSFormer Decoder.
    
    Extracts watermark from (attacked) watermarked image.
    Blind extraction - no access to original image.
    
    Args:
        in_channels: Input channels (3 for RGB)
        base_dim: Base channel dimension (C)
        num_stages: Number of downsampling stages (K)
        watermark_length: Watermark bit length (64)
    """
    def __init__(self, in_channels=3, base_dim=64, num_stages=3, watermark_length=64):
        super().__init__()
        self.num_stages = num_stages
        self.base_dim = base_dim
        self.watermark_length = watermark_length
        
        # Initial feature extraction
        # From paper: "we also use a 3×3 convolution to extract 
        # the shallow features"
        self.patch_embed = PatchEmbedding(in_channels, base_dim)
        
        # Downsampling stages with LCESTB
        self.down_blocks = nn.ModuleList()
        self.down_samples = nn.ModuleList()
        
        for i in range(num_stages):
            dim = base_dim * (2 ** i)
            
            # LCESTB blocks (W-MSA and SW-MSA)
            self.down_blocks.append(
                nn.Sequential(
                    LCESTB(dim, num_heads=4, window_size=8, shift_size=0),
                    LCESTB(dim, num_heads=4, window_size=8, shift_size=4)
                )
            )
            
            # Downsampling
            if i < num_stages - 1:
                self.down_samples.append(
                    nn.Conv2d(dim, dim * 2, kernel_size=4, stride=2, padding=1)
                )
        
        # Bottleneck with FETB
        bottleneck_dim = base_dim * (2 ** (num_stages - 1))
        self.bottleneck = FETB(bottleneck_dim, num_heads=4, num_blocks=2)
        
        # Information extraction layer
        # From paper: "I_output_no is passed through the information 
        # extraction layer, consisting of a convolutional layer and 
        # a fully connected layer"
        
        # Global pooling
        self.pool = nn.AdaptiveAvgPool2d(1)
        
        # FC layers for watermark extraction
        self.fc = nn.Sequential(
            nn.Linear(bottleneck_dim, bottleneck_dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(bottleneck_dim // 2, watermark_length),
            nn.Sigmoid()  # Output in [0, 1] for binary classification
        )
    
    def forward(self, x):
        """
        Args:
            x: (B, 3, H, W) - (attacked) watermarked image
        Returns:
            watermark: (B, L) - extracted watermark bits (continuous [0,1])
        """
        B, C, H, W = x.shape
        current_h, current_w = H, W
        
        # Initial features
        x = self.patch_embed(x)  # (B, base_dim, H, W)
        
        # Downsampling with LCESTB
        for i, (block, downsample) in enumerate(
            zip(self.down_blocks, self.down_samples + [None])
        ):
            x = block[0](x, current_h, current_w)
            x = block[1](x, current_h, current_w)
            
            if downsample is not None:
                x = downsample(x)
                current_h, current_w = current_h // 2, current_w // 2
        
        # Bottleneck
        x = self.bottleneck(x)  # (B, bottleneck_dim, H/2^K, W/2^K)
        
        # Global pooling
        x = self.pool(x)  # (B, bottleneck_dim, 1, 1)
        x = x.view(B, -1)  # (B, bottleneck_dim)
        
        # Extract watermark
        watermark = self.fc(x)  # (B, watermark_length)
        
        return watermark

# Test
print("Building decoder...")
decoder = Decoder(
    in_channels=3,
    base_dim=32,  # Reduced for memory
    num_stages=2,  # Reduced for memory
    watermark_length=64
).to(device)

# Count parameters
num_params = sum(p.numel() for p in decoder.parameters())
print(f"Decoder parameters: {num_params:,}")

# Test forward pass
with torch.no_grad():
    test_img = torch.randn(1, 3, 128, 128).to(device)
    extracted_wm = decoder(test_img)
    print(f"Input shape: {test_img.shape}")
    print(f"Extracted watermark shape: {extracted_wm.shape}")
    print(f"Watermark range: [{extracted_wm.min():.3f}, {extracted_wm.max():.3f}]")
    print(f"First 16 bits: {extracted_wm[0, :16].cpu().numpy()}")

print("✓ Decoder working!")

Building decoder...
Decoder parameters: 316,128
Input shape: torch.Size([1, 3, 128, 128])
Extracted watermark shape: torch.Size([1, 64])
Watermark range: [0.424, 0.563]
First 16 bits: [0.5145183  0.53554344 0.5155848  0.53028655 0.4664097  0.48691055
 0.46294343 0.46032423 0.42978856 0.5306961  0.49670574 0.4449556
 0.4426352  0.4966481  0.5147282  0.53451884]
✓ Decoder working!


In [19]:
class WatermarkLoss(nn.Module):
    """
    Combined loss for RoWSFormer training.
    
    Implements three loss components:
    1. Image fidelity (invisibility)
    2. Watermark recovery (robustness)
    3. Pixel constraint (validity)
    
    Args:
        lambda_1: Weight for image loss (default 2.0)
        lambda_2: Weight for watermark loss (default 10.0)
        lambda_3: Weight for constraint loss (default 0.1)
    """
    def __init__(self, lambda_1=2.0, lambda_2=10.0, lambda_3=0.1):
        super().__init__()
        self.lambda_1 = lambda_1
        self.lambda_2 = lambda_2
        self.lambda_3 = lambda_3
        
        self.mse = nn.MSELoss()
    
    def image_loss(self, cover, watermarked):
        """
        L_E: Image fidelity loss.
        Ensures watermark is imperceptible.
        
        Args:
            cover: Original image (B, 3, H, W)
            watermarked: Watermarked image (B, 3, H, W)
        """
        return self.mse(cover, watermarked)
    
    def watermark_loss(self, original_wm, extracted_wm):
        """
        L_D: Watermark recovery loss.
        Ensures watermark can be accurately extracted.
        
        Args:
            original_wm: Original watermark (B, L)
            extracted_wm: Extracted watermark (B, L)
        """
        return self.mse(original_wm, extracted_wm)
    
    def constraint_loss(self, watermarked):
        """
        L_C: Pixel constraint loss.
        Penalizes pixels outside [0, 1] range.
        
        From paper Eq. (6):
        L_C = Σ_{i,j} penalty(I_em[i,j])
        where penalty = 0.5 * |I - 1| if I > 1
                      = 0.5 * |I| if I < 0
                      = 0 otherwise
        
        Args:
            watermarked: Watermarked image (B, 3, H, W)
        """
        # Pixels > 1
        over = watermarked - 1.0
        over = torch.clamp(over, min=0)  # Only positive parts
        loss_over = 0.5 * over.abs().sum()
        
        # Pixels < 0
        under = watermarked
        under = torch.clamp(under, max=0)  # Only negative parts
        loss_under = 0.5 * under.abs().sum()
        
        # Normalize by number of pixels
        num_pixels = watermarked.numel()
        
        return (loss_over + loss_under) / num_pixels
    
    def forward(self, cover, watermarked, original_wm, extracted_wm):
        """
        Compute total loss.
        
        Args:
            cover: Original image (B, 3, H, W)
            watermarked: Watermarked image (B, 3, H, W)
            original_wm: Original watermark (B, L)
            extracted_wm: Extracted watermark (B, L)
        
        Returns:
            total_loss: Weighted sum of all losses
            loss_dict: Dictionary with individual loss values
        """
        # Compute individual losses
        L_E = self.image_loss(cover, watermarked)
        L_D = self.watermark_loss(original_wm, extracted_wm)
        L_C = self.constraint_loss(watermarked)
        
        # Total loss (Eq. 7)
        total_loss = self.lambda_1 * L_E + self.lambda_2 * L_D + self.lambda_3 * L_C
        
        # Return individual losses for logging
        loss_dict = {
            'total': total_loss.item(),
            'image': L_E.item(),
            'watermark': L_D.item(),
            'constraint': L_C.item()
        }
        
        return total_loss, loss_dict

# Test
criterion = WatermarkLoss(lambda_1=2.0, lambda_2=10.0, lambda_3=0.1)

# Simulate some outputs
test_cover = torch.rand(1, 3, 128, 128)
test_watermarked = test_cover + torch.randn_like(test_cover) * 0.01
test_wm_orig = torch.randint(0, 2, (1, 64), dtype=torch.float32)
test_wm_extracted = test_wm_orig + torch.randn(1, 64) * 0.1

total_loss, loss_dict = criterion(
    test_cover, test_watermarked, test_wm_orig, test_wm_extracted
)

print("Loss components:")
for key, value in loss_dict.items():
    print(f"  {key:12s}: {value:.6f}")

print("\n✓ Loss functions working!")

Loss components:
  total       : 0.084752
  image       : 0.000099
  watermark   : 0.008455
  constraint  : 0.000025

✓ Loss functions working!


In [22]:
!pip install tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 190.6 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 872.8 kB/s eta 0:00:00a 0:00:01


In [23]:
def train_epoch(encoder, decoder, noise_layer, dataloader, criterion, 
                optimizer, scaler, device, epoch):
    """
    Train for one epoch.
    
    Args:
        encoder: Encoder network
        decoder: Decoder network  
        noise_layer: Attack layer
        dataloader: Training data loader
        criterion: Loss function
        optimizer: Optimizer
        scaler: GradScaler for AMP
        device: Device to train on
        epoch: Current epoch number
    
    Returns:
        avg_losses: Dictionary of average losses
    """
    encoder.train()
    decoder.train()
    
    total_losses = {'total': 0, 'image': 0, 'watermark': 0, 'constraint': 0}
    num_batches = 0
    
    pbar = tqdm(dataloader, desc=f'Epoch {epoch}')
    for batch_idx, (images, watermarks) in enumerate(pbar):
        images = images.to(device)
        watermarks = watermarks.to(device)
        
        optimizer.zero_grad()
        
        # Mixed precision forward pass
        with autocast():
            # Encode watermark
            watermarked = encoder(images, watermarks)
            
            # Apply attack
            attacked = noise_layer(watermarked, images)
            
            # Decode watermark
            extracted = decoder(attacked)
            
            # Compute loss
            loss, loss_dict = criterion(
                images, watermarked, watermarks, extracted
            )
        
        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        # Accumulate losses
        for key in total_losses:
            total_losses[key] += loss_dict[key]
        num_batches += 1
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f"{loss_dict['total']:.4f}",
            'img': f"{loss_dict['image']:.4f}",
            'wm': f"{loss_dict['watermark']:.4f}"
        })
    
    # Average losses
    avg_losses = {k: v / num_batches for k, v in total_losses.items()}
    
    return avg_losses

# Test training components
print("Setting up training...")

# Initialize models
encoder_small = Encoder(base_dim=32, num_stages=2).to(device)
decoder_small = Decoder(base_dim=32, num_stages=2).to(device)
noise_layer = NoiseLayer()
criterion = WatermarkLoss(lambda_1=2.0, lambda_2=10.0, lambda_3=0.1)

# Optimizer
params = list(encoder_small.parameters()) + list(decoder_small.parameters())
optimizer = torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-4)

# Learning rate scheduler (cosine decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=100, eta_min=1e-6
)

# Gradient scaler for AMP
scaler = GradScaler()

print("✓ Training setup complete!")
print(f"\nTotal parameters: {sum(p.numel() for p in params):,}")
print(f"Encoder: {sum(p.numel() for p in encoder_small.parameters()):,}")
print(f"Decoder: {sum(p.numel() for p in decoder_small.parameters()):,}")

Setting up training...
✓ Training setup complete!

Total parameters: 890,067
Encoder: 573,939
Decoder: 316,128


/tmp/ipykernel_7282/94295680.py:90: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [24]:
# Main training loop
def train(encoder, decoder, noise_layer, train_loader, test_loader,
          criterion, optimizer, scheduler, scaler, device, num_epochs=10):
    """
    Full training loop.
    
    Args:
        encoder: Encoder network
        decoder: Decoder network
        noise_layer: Attack layer
        train_loader: Training data loader
        test_loader: Test data loader
        criterion: Loss function
        optimizer: Optimizer
        scheduler: Learning rate scheduler
        scaler: Gradient scaler
        device: Device
        num_epochs: Number of epochs
    """
    best_loss = float('inf')
    
    for epoch in range(num_epochs):
        # Train
        train_losses = train_epoch(
            encoder, decoder, noise_layer, train_loader,
            criterion, optimizer, scaler, device, epoch
        )
        
        # Update learning rate
        scheduler.step()
        
        # Print epoch summary
        print(f"\nEpoch {epoch} Summary:")
        print(f"  Total Loss: {train_losses['total']:.6f}")
        print(f"  Image Loss: {train_losses['image']:.6f}")
        print(f"  Watermark Loss: {train_losses['watermark']:.6f}")
        print(f"  Constraint Loss: {train_losses['constraint']:.6f}")
        print(f"  Learning Rate: {scheduler.get_last_lr()[0]:.6f}")
        
        # Save best model
        if train_losses['total'] < best_loss:
            best_loss = train_losses['total']
            torch.save({
                'epoch': epoch,
                'encoder_state_dict': encoder.state_dict(),
                'decoder_state_dict': decoder.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': best_loss,
            }, 'checkpoints/best_model.pth')
            print(f"  ✓ Saved best model (loss: {best_loss:.6f})")
        
        print("-" * 50)

print("Training function defined!")
print("\n💡 To actually train, run:")
print("train(encoder_small, decoder_small, noise_layer, train_loader,")
print("      test_loader, criterion, optimizer, scheduler, scaler, device, num_epochs=10)")

Training function defined!

💡 To actually train, run:
train(encoder_small, decoder_small, noise_layer, train_loader,
      test_loader, criterion, optimizer, scheduler, scaler, device, num_epochs=10)


In [26]:
def compute_psnr(img1, img2):
    """
    Compute PSNR between two images.
    
    Args:
        img1, img2: Images in range [0, 1], shape (B, C, H, W)
    
    Returns:
        psnr: PSNR value in dB (averaged over batch)
    """
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0:
        return float('inf')
    
    max_pixel = 1.0
    psnr = 10 * torch.log10(max_pixel ** 2 / mse)
    return psnr.item()

def compute_bit_accuracy(original, extracted, threshold=0.5):
    """
    Compute bit extraction accuracy.
    
    Args:
        original: Original watermark bits (B, L), values in {0, 1}
        extracted: Extracted watermark (B, L), values in [0, 1]
        threshold: Threshold for binary classification (default 0.5)
    
    Returns:
        accuracy: Percentage of correctly extracted bits
    """
    # Threshold extracted watermark to binary
    extracted_binary = (extracted > threshold).float()
    
    # Compute accuracy
    correct = (extracted_binary == original).float()
    accuracy = correct.mean().item() * 100
    
    return accuracy

# Test metrics
print("Testing evaluation metrics:")

# Perfect reconstruction
img1 = torch.rand(1, 3, 128, 128)
psnr_perfect = compute_psnr(img1, img1)
print(f"\nPerfect match PSNR: {psnr_perfect:.2f} dB (should be inf)")

# Small noise
img2 = img1 + torch.randn_like(img1) * 0.01
psnr_noisy = compute_psnr(img1, img2)
print(f"Small noise PSNR: {psnr_noisy:.2f} dB")

# Watermark accuracy
wm_orig = torch.randint(0, 2, (5, 64), dtype=torch.float32)
wm_extracted = wm_orig.clone()
wm_extracted[0, :10] = 1 - wm_extracted[0, :10]  # Flip 10 bits in first sample

acc = compute_bit_accuracy(wm_orig, wm_extracted)
print(f"\nBit accuracy with 10 errors: {acc:.2f}%")
print(f"Expected: ~96.9% (310/320 correct)")

# Random extraction (should be ~50%)
wm_random = torch.rand(5, 64)
acc_random = compute_bit_accuracy(wm_orig, wm_random)
print(f"Random extraction accuracy: {acc_random:.2f}% (should be ~50%)")

print("\n✓ Evaluation metrics working!")

Testing evaluation metrics:

Perfect match PSNR: inf dB (should be inf)
Small noise PSNR: 40.01 dB

Bit accuracy with 10 errors: 96.88%
Expected: ~96.9% (310/320 correct)
Random extraction accuracy: 51.88% (should be ~50%)

✓ Evaluation metrics working!
